# VARMA and VECM Models

This notebook extends VAR in two directions:

1. **VARMA(p,q)** — adds moving average terms to the VAR system,
   analogous to how ARMA extends AR in the univariate case
2. **VECM (Vector Error Correction Model)** — handles the case where
   series are non-stationary but *cointegrated* (they share a common
   stochastic trend)

Both models are less commonly used than plain VAR, but they fill
important gaps in the multivariate modelling toolkit.

Topics covered:
1. VARMA(p,q) theory and limitations
2. Fitting VARMA with `statsmodels.tsa.statespace.varmax.VARMAX`
3. When VARMA is useful vs when to stick with VAR
4. Cointegration — intuition and testing
5. VECM — theory and implementation
6. When to use VECM vs VAR

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
from statsmodels.tsa.stattools import adfuller

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")
rng = np.random.default_rng(42)

In [ ]:
def adf_summary(series, name=""):
    """Run the ADF test and print a compact summary."""
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pvalue = result[0], result[1]
    conclusion = "Stationary" if pvalue < 0.05 else "Non-stationary"
    print(f"{name:>25s}  ADF stat={stat:+.4f}  p={pvalue:.4f}  => {conclusion}")
    return pvalue < 0.05

---
## 1. VARMA(p,q) — Theory

The **VARMA(p,q)** model extends VAR by adding moving average terms:

$$
\mathbf{y}_t = \mathbf{c} + \mathbf{\Phi}_1 \mathbf{y}_{t-1} + \cdots + \mathbf{\Phi}_p \mathbf{y}_{t-p}
+ \boldsymbol{\varepsilon}_t + \mathbf{\Theta}_1 \boldsymbol{\varepsilon}_{t-1} + \cdots + \mathbf{\Theta}_q \boldsymbol{\varepsilon}_{t-q}
$$

where:
- $\mathbf{\Phi}_i$ are the $K \times K$ AR coefficient matrices (as in VAR)
- $\mathbf{\Theta}_j$ are the $K \times K$ MA coefficient matrices
- $\boldsymbol{\varepsilon}_t$ is the multivariate white noise vector

### Why VARMA is less common than VAR

1. **Identification problem**: different VARMA(p,q) specifications can produce
   the same autocovariance structure, making it hard to determine the "true" orders
2. **Estimation difficulty**: the likelihood surface is more complex, and optimization
   is more likely to fail or converge to local optima
3. **VAR approximation**: any stationary VARMA process can be approximated
   arbitrarily well by a VAR of sufficiently high order — so in practice, people
   just use a higher-order VAR

VARMA may be useful when:
- A low-order VARMA achieves parsimony (fewer parameters than the VAR approximation)
- You have strong theoretical reasons for MA dynamics

In [ ]:
# Load economic data for VARMA demonstration
m2 = pd.read_csv(DATA_DIR / "M2SLMoneyStock.csv", index_col="Date", parse_dates=True)
pce = pd.read_csv(DATA_DIR / "PCEPersonalSpending.csv", index_col="Date", parse_dates=True)

econ = pd.concat([m2, pce], axis=1).dropna()
econ.columns = ["M2", "PCE"]
econ.index.freq = pd.infer_freq(econ.index)

# Difference to achieve stationarity
econ_diff = econ.diff().dropna()
econ_diff.columns = ["dM2", "dPCE"]

# Train/test split
n_test = 24
train = econ_diff.iloc[:-n_test]
test = econ_diff.iloc[-n_test:]

print(f"Train: {len(train)} obs, Test: {len(test)} obs")

---
## 2. Fitting VARMA with `VARMAX`

The `statsmodels` class for VARMA is called `VARMAX` (it also supports
exogenous regressors, hence the X).  For a pure VARMA, simply omit the
`exog` parameter.

```python
from statsmodels.tsa.statespace.varmax import VARMAX
model = VARMAX(endog, order=(p, q))
result = model.fit(disp=False)
```

**Warning**: VARMA estimation can be slow and may encounter convergence
issues.  Start with low orders (p, q = 1 or 2).

In [ ]:
# Fit a VARMA(1,1) model
varma_model = VARMAX(train, order=(1, 1), trend="c")
varma_result = varma_model.fit(disp=False, maxiter=500)

print(varma_result.summary())

In [ ]:
# Forecast with VARMA(1,1)
varma_forecast = varma_result.forecast(steps=n_test)
varma_forecast.columns = ["dM2_varma", "dPCE_varma"]
varma_forecast.index = test.index

varma_forecast.head()

In [ ]:
# Compare VAR vs VARMA forecasts on differenced scale
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Also fit a plain VAR for comparison
var_model = VAR(train)
var_result = var_model.fit(maxlags=15, ic="aic")
var_forecast_arr = var_result.forecast(train.values[-var_result.k_ar:], steps=n_test)
var_forecast = pd.DataFrame(var_forecast_arr, index=test.index, columns=["dM2_var", "dPCE_var"])

print(f"VAR selected order: p = {var_result.k_ar}")
print()

for col_actual, col_var, col_varma in [
    ("dM2", "dM2_var", "dM2_varma"),
    ("dPCE", "dPCE_var", "dPCE_varma"),
]:
    actual = test[col_actual].values
    rmse_var = np.sqrt(mean_squared_error(actual, var_forecast[col_var].values))
    rmse_varma = np.sqrt(mean_squared_error(actual, varma_forecast[col_varma].values))
    print(f"{col_actual}:  VAR RMSE={rmse_var:.2f}  |  VARMA(1,1) RMSE={rmse_varma:.2f}")

print()
print("In many practical settings, VAR and VARMA produce similar results.")
print("The added complexity of VARMA often does not pay off.")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, (col_a, col_v, col_vm) in enumerate([
    ("dM2", "dM2_var", "dM2_varma"),
    ("dPCE", "dPCE_var", "dPCE_varma"),
]):
    axes[i].plot(test[col_a], label="Actual", color="black", linewidth=2)
    axes[i].plot(var_forecast[col_v], label="VAR", linestyle="--", color="tab:blue")
    axes[i].plot(varma_forecast[col_vm], label="VARMA(1,1)", linestyle=":", color="tab:red")
    axes[i].set_title(col_a)
    axes[i].legend()

plt.suptitle("VAR vs VARMA(1,1) — Forecast Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### When to use VARMA vs VAR

| Consideration | VAR | VARMA |
|---|---|---|
| Ease of estimation | Simple, reliable | Can be fragile |
| Identification | Straightforward | Identification problems |
| Parsimony | May need many lags | Potentially fewer parameters |
| Software support | Excellent | Limited |
| Practical advice | **Default choice** | Use only with good reason |

**Bottom line**: start with VAR.  Only consider VARMA if you have strong
theoretical motivation and can demonstrate improved out-of-sample performance.

---
## 3. Cointegration — When Non-Stationary Series Share a Common Trend

Two I(1) (first-difference-stationary) series $y_{1,t}$ and $y_{2,t}$ are
**cointegrated** if there exists a linear combination
$\beta_1 y_{1,t} + \beta_2 y_{2,t}$ that is stationary.

Intuitively, cointegrated series may wander individually (each has a
unit root), but they are "tied together" by a long-run equilibrium
relationship.  When they drift apart, an error-correction mechanism
pulls them back together.

**Examples**:
- Spot and futures prices of the same commodity
- Interest rates at different maturities
- Consumer income and spending

### Why cointegration matters for VAR

If you difference cointegrated series and fit a VAR, you **lose
information** about the long-run relationship.  The correct model
is a **VECM**, which models the short-run dynamics while respecting
the cointegrating relationship.

In [ ]:
# Simulate two cointegrated series
n = 300
eps1 = rng.standard_normal(n)
eps2 = rng.standard_normal(n)

# Common stochastic trend (random walk)
trend = np.cumsum(rng.standard_normal(n))

# Two series that share the common trend but have different noise
y1 = trend + 2 * np.cumsum(eps1 * 0.1)  # mostly follows the trend
y2 = 0.5 * trend + np.cumsum(eps2 * 0.1)  # follows the trend with different loading

coint_df = pd.DataFrame({"y1": y1, "y2": y2})

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y1, label="$y_1$ (cointegrated)", alpha=0.8)
ax.plot(y2, label="$y_2$ (cointegrated)", alpha=0.8)
ax.set_title("Two Cointegrated Series (Simulated)")
ax.set_xlabel("t")
ax.legend()
plt.tight_layout()
plt.show()

print("Both series wander (non-stationary individually),")
print("but they move together due to the shared stochastic trend.")

In [ ]:
# ADF tests confirm each series is non-stationary
print("ADF tests on individual series:")
adf_summary(coint_df["y1"], "y1")
adf_summary(coint_df["y2"], "y2")

print()

# But their linear combination may be stationary
# Simple OLS: regress y1 on y2 and check the residual
from numpy.polynomial.polynomial import polyfit
beta = np.polyfit(y2, y1, 1)
spread = y1 - beta[0] * y2 - beta[1]

print("ADF test on the linear combination (spread = y1 - beta * y2):")
adf_summary(pd.Series(spread), "spread")

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(spread, linewidth=0.8)
ax.axhline(0, color="red", linestyle="--", alpha=0.5)
ax.set_title("Cointegrating Spread: $y_1 - \\hat{\\beta} y_2$")
ax.set_xlabel("t")
plt.tight_layout()
plt.show()

print("The spread is (approximately) stationary — confirming cointegration.")

---
## 4. Johansen Cointegration Test

The **Johansen test** is the standard multivariate test for cointegration.
Unlike the Engle-Granger two-step approach (regress, then test residual),
Johansen tests for the number of cointegrating relationships directly
and works for systems with more than two variables.

It produces two test statistics:
- **Trace statistic**: tests H0 that there are at most $r$ cointegrating vectors
- **Maximum eigenvalue statistic**: tests $r$ vs $r+1$ cointegrating vectors

```python
from statsmodels.tsa.vector_ar.vecm import coint_johansen
result = coint_johansen(data, det_order=0, k_ar_diff=1)
```

Parameters:
- `det_order`: -1 (no deterministic terms), 0 (constant), 1 (trend)
- `k_ar_diff`: number of lagged difference terms

In [ ]:
# Johansen test on our simulated cointegrated data
johansen_result = coint_johansen(coint_df, det_order=0, k_ar_diff=1)

print("Johansen Cointegration Test")
print("=" * 60)
print()
print("Trace Statistic:")
print(f"  {'r':>5s}  {'Trace Stat':>12s}  {'90%':>8s}  {'95%':>8s}  {'99%':>8s}")
for i in range(len(johansen_result.lr1)):
    print(
        f"  r<={i:d}  {johansen_result.lr1[i]:12.4f}"
        f"  {johansen_result.cvt[i, 0]:8.4f}"
        f"  {johansen_result.cvt[i, 1]:8.4f}"
        f"  {johansen_result.cvt[i, 2]:8.4f}"
    )

print()
print("Max Eigenvalue Statistic:")
print(f"  {'r':>5s}  {'Max Eig':>12s}  {'90%':>8s}  {'95%':>8s}  {'99%':>8s}")
for i in range(len(johansen_result.lr2)):
    print(
        f"  r<={i:d}  {johansen_result.lr2[i]:12.4f}"
        f"  {johansen_result.cvm[i, 0]:8.4f}"
        f"  {johansen_result.cvm[i, 1]:8.4f}"
        f"  {johansen_result.cvm[i, 2]:8.4f}"
    )

print()
print("If the test statistic exceeds the critical value, reject H0.")
print("Rejecting r<=0 but not r<=1 indicates exactly 1 cointegrating relationship.")

In [ ]:
# Johansen test on the real economic data (M2 and PCE in levels)
print("Johansen test on M2 and PCE (levels):")
print("=" * 60)

johansen_econ = coint_johansen(econ, det_order=0, k_ar_diff=2)

print()
print("Trace Statistic:")
print(f"  {'r':>5s}  {'Trace Stat':>12s}  {'90%':>8s}  {'95%':>8s}  {'99%':>8s}")
for i in range(len(johansen_econ.lr1)):
    print(
        f"  r<={i:d}  {johansen_econ.lr1[i]:12.4f}"
        f"  {johansen_econ.cvt[i, 0]:8.4f}"
        f"  {johansen_econ.cvt[i, 1]:8.4f}"
        f"  {johansen_econ.cvt[i, 2]:8.4f}"
    )

print()
print("If M2 and PCE are cointegrated, a VECM may be more appropriate")
print("than differencing and fitting a plain VAR.")

---
## 5. VECM — Vector Error Correction Model

When $K$ series are cointegrated with $r$ cointegrating relationships,
the VECM representation is:

$$
\Delta \mathbf{y}_t = \boldsymbol{\alpha} \boldsymbol{\beta}' \mathbf{y}_{t-1}
+ \mathbf{\Gamma}_1 \Delta \mathbf{y}_{t-1} + \cdots + \mathbf{\Gamma}_{p-1} \Delta \mathbf{y}_{t-p+1}
+ \mathbf{c} + \boldsymbol{\varepsilon}_t
$$

where:
- $\boldsymbol{\beta}$ is the $K \times r$ matrix of cointegrating vectors
- $\boldsymbol{\alpha}$ is the $K \times r$ matrix of **adjustment coefficients**
  (how fast each variable adjusts to disequilibrium)
- $\boldsymbol{\alpha} \boldsymbol{\beta}' \mathbf{y}_{t-1}$ is the **error correction term** —
  it pulls the system back toward equilibrium when series drift apart
- $\mathbf{\Gamma}_i$ capture short-run dynamics (like a VAR on differences)

The key insight: VECM separates the **long-run equilibrium** (cointegrating
relationship) from the **short-run dynamics** (how the system adjusts).

In [ ]:
# Fit VECM to the simulated cointegrated data
vecm_sim = VECM(coint_df, k_ar_diff=1, coint_rank=1, deterministic="ci")
vecm_sim_result = vecm_sim.fit()

print(vecm_sim_result.summary())

In [ ]:
# The cointegrating vector beta
print("Cointegrating vector (beta):")
print(vecm_sim_result.beta)
print()
print("Adjustment coefficients (alpha):")
print(vecm_sim_result.alpha)
print()
print("The alpha coefficients show how quickly each variable adjusts")
print("when the system deviates from the long-run equilibrium.")
print("Negative alpha for a variable means it moves back toward equilibrium.")

In [ ]:
# Forecast with VECM
vecm_forecast = vecm_sim_result.predict(steps=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(["y1", "y2"]):
    axes[i].plot(coint_df[col].iloc[-80:].values, label="Observed", color="black")
    forecast_idx = range(len(coint_df[col].iloc[-80:]), len(coint_df[col].iloc[-80:]) + 30)
    axes[i].plot(forecast_idx, vecm_forecast[:, i], label="VECM Forecast",
                 color="tab:red", linestyle="--")
    axes[i].axvline(len(coint_df[col].iloc[-80:]) - 1, color="grey", linestyle=":", alpha=0.7)
    axes[i].set_title(f"VECM Forecast — {col}")
    axes[i].legend()

plt.tight_layout()
plt.show()

print("VECM forecasts preserve the cointegrating relationship:")
print("the forecasted series stay linked, unlike separate random walks.")

In [ ]:
# Fit VECM to the economic data (if cointegrated)
try:
    vecm_econ = VECM(econ, k_ar_diff=2, coint_rank=1, deterministic="ci")
    vecm_econ_result = vecm_econ.fit()
    print(vecm_econ_result.summary())
except Exception as e:
    print(f"VECM fitting failed: {e}")
    print("This may indicate the series are not cointegrated,")
    print("or the model specification needs adjustment.")

---
## 6. When to Use Which Model

| Series properties | Recommended model |
|---|---|
| All stationary | **VAR** on levels |
| Non-stationary, **not** cointegrated | **VAR** on differences |
| Non-stationary, **cointegrated** | **VECM** on levels |
| Need MA dynamics | **VARMA** (but try VAR first) |

### Decision flowchart

1. Test each series for stationarity (ADF)
2. If stationary: fit VAR on levels
3. If non-stationary:
   a. Test for cointegration (Johansen)
   b. If cointegrated: fit VECM
   c. If not cointegrated: difference and fit VAR
4. Only consider VARMA if VAR residuals show strong MA patterns and
   you need parsimony

In [ ]:
print("Key takeaways:")
print()
print("1. VARMA adds MA terms to VAR but is harder to estimate and identify.")
print("   In practice, a higher-order VAR usually works just as well.")
print()
print("2. Cointegration means non-stationary series share a common trend.")
print("   Differencing cointegrated series loses the long-run information.")
print()
print("3. VECM respects the cointegrating relationship while modelling")
print("   short-run dynamics. Use it when Johansen test finds cointegration.")
print()
print("4. Default strategy: start with VAR. Only move to VECM or VARMA")
print("   when there is clear evidence that VAR is inadequate.")

---
## Summary

- **VARMA(p,q)** extends VAR with MA terms: more flexible in theory,
  but identification problems and estimation difficulty make it less
  practical.  Use `VARMAX(endog, order=(p,q))` in statsmodels.

- **Cointegration** occurs when non-stationary series share a common
  stochastic trend.  Test with `coint_johansen()`.

- **VECM** is the correct model for cointegrated series.  It decomposes
  dynamics into a long-run equilibrium (cointegrating vector $\boldsymbol{\beta}$)
  and short-run adjustment (loading matrix $\boldsymbol{\alpha}$).

- **Practical hierarchy**: VAR (default) > VECM (if cointegrated) > VARMA (rarely needed)

In [ ]:
print("Next notebook: practical multivariate forecasting — end-to-end pipeline")
print("and honest comparison with univariate ARIMA.")